# AIoT Helmet / Triple-Riding Detection: Training Notebook

Trains **Option B** (our own detector) and the **YOLO26** comparison baseline on both tasks, then scores all four on the same held-out test split.

Runs in two places with no code changes:
- **Local** (project `.venv` or any Jupyter kernel with a CUDA or CPU PyTorch install)
- **Google Colab** (uses a Google Drive folder, see Section 1)

Every step calls the scripts in `training/`, so the notebook and the command line produce the same results.

## 1. Locate the project and detect the environment

In [ ]:
import os, sys
from pathlib import Path

os.environ["PYTHONWARNINGS"] = "ignore"  # applies to the training subprocesses started below
import warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # The project lives in My Drive/Parin: training/, dataset/ (or dataset.zip) and all outputs.
    ROOT = Path("/content/drive/MyDrive/Parin")
    if not (ROOT / "training" / "config.py").exists():
        raise SystemExit(f"{ROOT / 'training'} not found. Upload the training/ folder to My Drive/Parin first.")
else:
    here = Path.cwd()
    ROOT = here.parent if (here / "config.py").exists() else here

TRAINING = ROOT / "training"
assert (TRAINING / "config.py").exists(), f"training/ not found under {ROOT}. Fix ROOT above."
if IN_COLAB and "/content/drive" not in str(TRAINING.resolve()):
    raise SystemExit("On Colab, ROOT must be on Drive so checkpoints survive a dropped session.")
sys.path.insert(0, str(TRAINING))

if IN_COLAB:
    # Reading thousands of small images straight from the Drive mount is very slow (about 10 s per batch on an L4).
    # Copy the dataset once to the Colab disk; checkpoints and logs stay on Drive.
    import shutil, time
    local_dataset = Path("/content/dataset")
    marker = local_dataset / ".copied"
    if not marker.exists():
        zip_path = ROOT / "dataset.zip"
        if not zip_path.exists():
            raise SystemExit(f"{zip_path} not found. Upload dataset.zip (made from the local dataset/ folder) to {ROOT}.")
        t0 = time.time()
        print("Unzipping dataset.zip to /content (one time, about a minute)...")
        import zipfile
        staging = Path("/content/dataset_staging")
        shutil.rmtree(staging, ignore_errors=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(staging)
        # The zip may hold the folders at its top level or inside one extra folder (dataset/). Find where they are.
        dataset_root = [d for d in [staging, *staging.rglob("*")] if d.is_dir()
                        and (d / "helmet_and_plate" / "data.yaml").exists() and (d / "triple_riding" / "data.yaml").exists()]
        if len(dataset_root) != 1:
            raise SystemExit(f"dataset.zip must contain helmet_and_plate/ and triple_riding/ (found under: {dataset_root}). "
                             "Re-zip the contents of dataset/, not a different folder.")
        shutil.rmtree(local_dataset, ignore_errors=True)
        shutil.move(str(dataset_root[0]), str(local_dataset))
        shutil.rmtree(staging, ignore_errors=True)
        marker.touch()
        print(f"unzipped in {time.time() - t0:.0f} s")
    os.environ["AIOT_DATASET_DIR"] = str(local_dataset)  # config.py and the training scripts read the dataset from here

print("IN_COLAB:", IN_COLAB)
print("ROOT:    ", ROOT)
print("DATASET: ", os.environ.get("AIOT_DATASET_DIR", ROOT / "dataset"))

## 2. Install dependencies

Torch is already present on Colab. Locally, install it first if you want GPU (see `training/requirements.txt`).

In [ ]:
%pip install -q torchmetrics pycocotools ultralytics pyyaml

## 3. Check hardware and dataset

In [ ]:
import torch
from config import TASKS, pick_device
from yolo_data import load_data_yaml

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected: training will run on CPU and be very slow. Use Colab T4 or a CUDA machine.")

for task, cfg in TASKS.items():
    names, splits = load_data_yaml(cfg["data"])
    counts = {s: len(list(p.iterdir())) for s, p in splits.items()}
    print(f"{task}: classes={names} images={counts}")

## 4. Settings

These are the defaults from `training/config.py`. Change them here for a quick run (e.g. `EPOCHS = 5` to smoke-test).

In [ ]:
EPOCHS = 50
IMGSZ = 640
BATCH = 16
PATIENCE = 10
FREEZE_EPOCHS = 10   # Option B: backbone frozen for the first N epochs
WORKERS = min(4, os.cpu_count() or 2) if IN_COLAB else 4  # data loading uses the CPU; Colab reports its core count
DEVICE = ""          # "" = auto, "0" = first GPU, "cpu"
LIMIT = None         # set e.g. 200 to use only the first 200 images per split (quick test)
from config import RUNS_DIR

def resume_flag(run):
    # Continue a run from its last saved epoch if one exists on Drive (after a dropped session); otherwise start fresh.
    return "--resume" if (RUNS_DIR / run / "weights" / "last.pt").exists() else ""

## 5. Train Option B (our own detector), helmet + plate (Model A)

In [ ]:
import subprocess
extra = f"--limit {LIMIT}" if LIMIT else ""
resume = resume_flag("helmet_plate_optionb")
!{sys.executable} -u "{TRAINING / 'train_option_b.py'}" --task helmet_plate --epochs {EPOCHS} --imgsz {IMGSZ} --batch {BATCH} --patience {PATIENCE} --freeze-epochs {FREEZE_EPOCHS} --workers {WORKERS} --device "{DEVICE}" {extra} {resume}

## 6. Train Option B, triple riding (Model B)

In [ ]:
extra = f"--limit {LIMIT}" if LIMIT else ""
resume = resume_flag("triple_riding_optionb")
!{sys.executable} -u "{TRAINING / 'train_option_b.py'}" --task triple_riding --epochs {EPOCHS} --imgsz {IMGSZ} --batch {BATCH} --patience {PATIENCE} --freeze-epochs {FREEZE_EPOCHS} --workers {WORKERS} --device "{DEVICE}" {extra} {resume}

## 7. Train YOLO26 comparison baseline, helmet + plate

In [ ]:
!{sys.executable} -u "{TRAINING / 'train_yolo26.py'}" --task helmet_plate --weights yolo26s.pt --epochs {EPOCHS} --imgsz {IMGSZ} --batch {BATCH} --patience {PATIENCE} --workers {WORKERS} --device "{DEVICE}" {resume_flag("yolo26/helmet_plate")}

## 8. Train YOLO26 comparison baseline, triple riding

In [ ]:
!{sys.executable} -u "{TRAINING / 'train_yolo26.py'}" --task triple_riding --weights yolo26n.pt --epochs {EPOCHS} --imgsz {IMGSZ} --batch {BATCH} --patience {PATIENCE} --workers {WORKERS} --device "{DEVICE}" {resume_flag("yolo26/triple_riding")}

In [ ]:
from config import RUNS_DIR
# Confirm every checkpoint is on Drive (or local disk), with its size and last-modified time.
import datetime
for p in sorted(RUNS_DIR.rglob("*.pt")) + sorted(RUNS_DIR.rglob("*.csv")):
    ts = datetime.datetime.fromtimestamp(p.stat().st_mtime).strftime("%Y-%m-%d %H:%M")
    print(f"{p.relative_to(ROOT)}  {p.stat().st_size / 1e6:8.1f} MB  {ts}")

## 9. Head-to-head test scores

Both models are scored on the **test** split with the same pixels and the same mAP code (`training/evaluate.py`).

In [ ]:
import json
from config import RUNS_DIR, TASKS

YOLO_WEIGHTS = {t: RUNS_DIR / "yolo26" / t / "weights" / "best.pt" for t in TASKS}
OPTB_WEIGHTS = {t: RUNS_DIR / f"{t}_optionb" / "weights" / "best.pt" for t in TASKS}

summary = {}
for task in TASKS:
    out = RUNS_DIR / f"compare_{task}.json"
    !{sys.executable} -u "{TRAINING / 'evaluate.py'}" --task {task} --split test --option-b "{OPTB_WEIGHTS[task]}" --yolo "{YOLO_WEIGHTS[task]}" --device "{DEVICE}" --out "{out}"
    summary[task] = json.loads(out.read_text())

print(f"{'task':<16}{'model':<12}{'mAP50':>8}{'mAP50-95':>10}{'recall@100':>12}")
for task, res in summary.items():
    for key, label in (("option_b", "Option B"), ("yolo26", "YOLO26")):
        if key in res:
            m = res[key]
            print(f"{task:<16}{label:<12}{m['mAP50']:>8.3f}{m['mAP50_95']:>10.3f}{m['recall_100']:>12.3f}")

## 10. Where the weights are

- Option B: `training/runs/<task>_optionb/weights/best.pt` (plus `last.pt`, `log.csv`, `results.json`)
- YOLO26: `training/runs/yolo26/<task>/weights/best.pt`
- Comparison tables: `training/runs/compare_<task>.json`

On Colab, the checkpoints are written straight to Drive, so they survive a dropped session. If the session drops, reopen the notebook, run all cells again. Each training cell continues from its last saved epoch on Drive (or starts fresh if there is none), and finished runs are skipped.

## 11. Analysis: charts, error rates and logs

Reads the logs of all finished runs and writes them to the **`analysis/`** folder in the project (on Drive when on Colab):
- `epoch_log.csv`: every epoch of every model (learning rate, loss, mAP, error rate, seconds per epoch)
- `summary.txt`: error rate from start to best, time per epoch, test scores
- `error_rate.png`, `val_map.png`, `loss.png`, `learning_rate.png`, `epoch_time.png`, `test_comparison.png`

It also zips the folder to `analysis.zip` and downloads it on Colab.

In [ ]:
import shutil
import subprocess
from IPython.display import Image, display
from config import ANALYSIS_DIR

subprocess.run([sys.executable, "-u", str(TRAINING / "analyze.py")], check=True)
for png in sorted(ANALYSIS_DIR.glob("*.png")):
    display(Image(str(png)))
print((ANALYSIS_DIR / "summary.txt").read_text(encoding="utf-8"))

archive = shutil.make_archive(str(ROOT / "analysis"), "zip", ANALYSIS_DIR)
print("saved:", ANALYSIS_DIR)
print("zip:  ", archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)